In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
import time
import re

import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [3]:
DATABASE_NAME = "internet_governance_news.db"

def create_database():
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS articles (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT,
            date TEXT,
            author TEXT,
            url TEXT UNIQUE,
            source TEXT
        )
    """)
    conn.commit()
    conn.close()
    print("✅ Banco e tabela 'articles' prontos!")

create_database()

✅ Banco e tabela 'articles' prontos!


In [4]:
def insert_article(title, date, author, url, source):
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    try:
        cursor.execute("""
            INSERT INTO articles (title, date, author, url, source)
            VALUES (?, ?, ?, ?, ?)
        """, (title, date, author, url, source))
        conn.commit()
        return True
    except sqlite3.IntegrityError:
        return False
    finally:
        conn.close()

In [5]:
url = "https://icn.gob.ar/noticias"
print(f"📄 Coletando página única de notícias: {url}")

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

r = requests.get(url, headers=headers, timeout=10)
soup = BeautifulSoup(r.text, "html.parser")

noticias = []
paineis = soup.find_all("div", class_="panel")

if not paineis:
    print("⚠️ Nenhuma lista encontrada")
else:
    print(f"   {len(paineis)} notícias encontradas")

    for item in paineis:
        slug = item.get("data-slug")
        if not slug:
            continue
            
        link = f"https://icn.gob.ar/noticias/{slug}"
        
        conteudo = item.find("div", class_="contenido-panel")
        if not conteudo:
            continue
            
        titulo_tag = conteudo.find("h4")
        data_tag = conteudo.find("time")

        if not titulo_tag:
            continue

        data = {
            "title": titulo_tag.get_text(strip=True),
            "date": data_tag.get_text(strip=True) if data_tag else None,
            "author": "ICN",
            "url": link,
            "source": "Imprenta del Congreso (ICN)"
        }

        noticias.append(data)
        insert_article(**data)

print(f"\n✅ Total coletado na ICN: {len(noticias)} notícias")
df_icn = pd.DataFrame(noticias)
if not df_icn.empty:
    display(df_icn.head())

📄 Coletando página única de notícias: https://icn.gob.ar/noticias
   51 notícias encontradas

✅ Total coletado na ICN: 51 notícias


,title,date,author,url,source
0,Inauguración del período de sesiones ordinaria...,1 de marzo de 2024,ICN,https://icn.gob.ar/noticias/inauguracion-del-p...,Imprenta del Congreso (ICN)
1,Nuevos espacios de lactancia en la Imprenta de...,8 de noviembre de 2023,ICN,https://icn.gob.ar/noticias/nuevos-espacios-de...,Imprenta del Congreso (ICN)
2,Presentación del libro “Democracia 40 años” en...,19 de octubre de 2023,ICN,https://icn.gob.ar/noticias/presentacion-del-l...,Imprenta del Congreso (ICN)
3,Gestión de residuos sólidos urbanos,17 de octubre de 2023,ICN,https://icn.gob.ar/noticias/gestion-de-residuo...,Imprenta del Congreso (ICN)
4,La Imprenta del Congreso en La Noche de los Mu...,18 de septiembre de 2023,ICN,https://icn.gob.ar/noticias/la-imprenta-del-co...,Imprenta del Congreso (ICN)


In [6]:
def load_articles():
    conn = sqlite3.connect(DATABASE_NAME)
    df = pd.read_sql("""
        SELECT * FROM articles
        ORDER BY date DESC
    """, conn)
    conn.close()
    return df

df_db = load_articles()
print(f"📦 Total no banco: {len(df_db)} registros")
display(df_db.head(20))

📦 Total no banco: 51 registros


,id,title,date,author,url,source
0,2,Nuevos espacios de lactancia en la Imprenta de...,8 de noviembre de 2023,ICN,https://icn.gob.ar/noticias/nuevos-espacios-de...,Imprenta del Congreso (ICN)
1,24,Los colores del Congreso,8 de junio de 2021,ICN,https://icn.gob.ar/noticias/los-colores-del-co...,Imprenta del Congreso (ICN)
2,48,"Participación institucional en la jornada ""La ...",7 de octubre de 2019,ICN,https://icn.gob.ar/noticias/participacion-inst...,Imprenta del Congreso (ICN)
3,41,Compromiso para fortalecer la igualdad de géne...,7 de julio de 2020,ICN,https://icn.gob.ar/noticias/compromiso-para-fo...,Imprenta del Congreso (ICN)
4,20,El director del Museo Malvinas visitó la ICN,6 de agosto de 2021,ICN,https://icn.gob.ar/noticias/el-director-del-mu...,Imprenta del Congreso (ICN)
5,34,Cirelli: “Hoy la prioridad es el cuidado de la...,5 de octubre de 2020,ICN,https://icn.gob.ar/noticias/cirelli-hoy-la-pri...,Imprenta del Congreso (ICN)
6,32,"Luz Aimee Díaz, la primera trabajadora trans d...",5 de noviembre de 2020,ICN,https://icn.gob.ar/noticias/luz-aimee-diaz-la-...,Imprenta del Congreso (ICN)
7,11,Presentación del libro “Ley de derecho de acce...,5 de mayo de 2023,ICN,https://icn.gob.ar/noticias/presentacion-del-l...,Imprenta del Congreso (ICN)
8,17,"La Imprenta del Congreso, presente en Tecnópol...",4 de noviembre de 2021,ICN,https://icn.gob.ar/noticias/la-imprenta-del-co...,Imprenta del Congreso (ICN)
9,25,Gestión de calidad: un compromiso de todos y t...,4 de mayo de 2021,ICN,https://icn.gob.ar/noticias/gestion-de-calidad...,Imprenta del Congreso (ICN)


In [8]:
def plot_charts(df):
    if df.empty:
        print("❌ Sem dados para plotar")
        return

    # Top 15
    top15 = df.head(15).copy()
    top15["rank"] = range(1, len(top15) + 1)

    fig1 = px.bar(
        top15,
        x="rank",
        y="title",
        orientation="h",
        title="Top 15 Notícias – Internet Governance"
    )
    fig1.update_layout(height=600)
    fig1.show()

    # Fonte
    source_count = df["source"].value_counts().reset_index()
    source_count.columns = ["source", "count"]

    fig2 = px.pie(
        source_count,
        names="source",
        values="count",
        title="Distribuição por Fonte"
    )
    fig2.show()

    # Palavras
    text = " ".join(df["title"].astype(str)).lower()
    words = re.findall(r"\b\w{4,}\b", text)

    word_freq = (
        pd.Series(words)
        .value_counts()
        .head(20)
        .reset_index()
    )
    word_freq.columns = ["palavra", "freq"]

    fig3 = px.treemap(
        word_freq,
        path=["palavra"],
        values="freq",
        title="Palavras mais frequentes nos títulos"
    )
    fig3.show()

plot_charts(df_db)